In [2]:
import pandas as pd
import numpy as np
from itertools import combinations

In [4]:
def calculate_gini(series:pd.Series)->float:
    if len(series)==0:
        return 0.0
    p_k=series.value_counts(normalize=True)
    return float(1-np.sum(p_k*p_k))

def calculate_split_gini(y_left:pd.Series,y_right:pd.Series)->float:
    n_left=len(y_left)
    n_right=len(y_right)
    n_total=n_left+n_right

    if n_total==0:
        return 0.0
    gini_left=calculate_gini(y_left)
    gini_right=calculate_gini(y_right)

    return float(
        (n_left/n_total)*gini_left+(n_right/n_total)*gini_right
    )

def get_binary_combinations(unique_values):
    """为离散特征生成所有可能的二元切分组合（严格去重版）"""
    res = []
    unique_values = list(unique_values)
    n = len(unique_values)
    
    if n <= 1:
        return res
        
    # 1. 对于小于总数一半的 r，生成的组合绝对不会和另一半重复
    for r in range(1, n // 2):
        for comb in combinations(unique_values, r):
            res.append(list(comb))
            
    # 2. 当 r 刚好等于总数一半时（且 n 是偶数）
    if n % 2 == 0:
        first_elem = unique_values[0]
        rest_elems = unique_values[1:]
        # 强行让第一个元素固定在左边，从剩下的里面选 (n//2 - 1) 个
        for comb in combinations(rest_elems, (n // 2) - 1):
            res.append([first_elem] + list(comb))
    else:
        # 如果 n 是奇数，r = n // 2 那一层也是绝对不会重复的
        for comb in combinations(unique_values, n // 2):
            res.append(list(comb))
            
    return res

class Node:
    def __init__(
        self,
        feature=None,
        split_point=None,
        left=None,
        right=None,
        value=None,
        is_leaf=False
    ):
        self.feature=feature
        self.split_point=split_point
        self.left=left
        self.right=right
        self.value=value
        self.is_leaf=is_leaf

class CARTTreeClassifier:
    def __init__(
        self,
        max_depth=None,
        min_samples_split=2,
        min_impurity_decrease=0.0
    ):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_impurity_decrease = min_impurity_decrease
        self.root = None
        self.continuous_cols=set()
        self.target_col = None
    def fit(self,df,target_col,feature_cols,continuous_cols=None):
        self.continuous_cols=set(continuous_cols)if continuous_cols is not None else set()
        self.target_col=target_col
        features=feature_cols
        self.root=self._build_tree(df,features,depth=0)
        return self
    def _build_tree(self,df,features,depth):
        y=df[self.target_col]
        num_samples=len(df)
        num_labels=len(y.unique())
        #预剪枝
        if (num_labels == 1 or 
            (self.max_depth is not None and depth >= self.max_depth) or 
            num_samples < self.min_samples_split or 
            len(features) == 0):
            return Node(value=y.mode()[0], is_leaf=True)

        best_gini=float('inf')
        best_feature=None
        best_split_point=None
        current_gini=calculate_gini(y)
        for feature in features:
            unique_vals=df[feature].unique()
            if feature in self.continuous_cols:
                sorted_vals=np.sort(unique_vals)
                split_candidates=(sorted_vals[:-1]+sorted_vals[1:])/2.0
                for split_point in split_candidates:
                    left_mask=df[feature]<=split_point
                    right_mask=~left_mask
                    score=calculate_split_gini(
                        df.loc[left_mask,self.target_col],df.loc[right_mask,self.target_col]
                    )
                    if score<best_gini:
                        best_gini,best_feature,best_split_point=score,feature,split_point
            else:
                split_candidates=get_binary_combinations(unique_vals)
                for split_point in split_candidates:
                    left_mask=df[feature].isin(split_point)
                    right_mask=~left_mask
                    score=calculate_split_gini(
                        df.loc[left_mask,self.target_col],df.loc[right_mask,self.target_col]
                    )
                    if score<best_gini:
                        best_gini,best_feature,best_split_point=score,feature,split_point
        if current_gini-best_gini<self.min_impurity_decrease or best_feature is None:
            return Node(value=y.mode()[0],is_leaf=True)

        if best_feature in self.continuous_cols:
            left_mask=df[best_feature]<=best_split_point
        else:
            left_mask=df[best_feature].isin(best_split_point)
        right_mask=~left_mask

        left_df=df[left_mask]
        right_df=df[right_mask]
        left_child=self._build_tree(left_df,features,depth+1) 
        right_child=self._build_tree(right_df,features,depth+1)
        return Node(feature=best_feature,split_point=best_split_point,
                   left=left_child,right=right_child)

    def predict(self,df):
        return df.apply(self._predict_row,axis=1)

    def _predict_row(self,row,node=None):
        if node is None:
            node=self.root
        if node.is_leaf:
            return node.value
        feat_val=row[node.feature]
        if node.feature in self.continuous_cols:
            if feat_val<=node.split_point:
                return self._predict_row(row,node.left)
            else:
                return self._predict_row(row,node.right)
        else:
            if feat_val in node.split_point:
                return self._predict_row(row,node.left)
            else:
                return self._predict_row(row,node.right)

In [19]:
df_train=pd.read_csv("c45_loan_train.csv")
df_test=pd.read_csv("c45_loan_test.csv")
df=df_train.copy()

x_cols=df.drop(['id','approved'],axis=1).columns.to_list()
c_cols=['age','income_k','credit_score','debt_ratio']
y_col='approved'
df.head()

,id,age,income_k,credit_score,debt_ratio,employment,education,has_house,loan_purpose,approved
0,TR001,29,11.4,640,0.39,full_time,high_school,yes,car,yes
1,TR002,42,37.6,709,0.45,self_employed,bachelor,no,business,yes
2,TR003,25,44.8,536,0.15,full_time,bachelor,yes,car,yes
3,TR004,22,12.8,557,0.66,part_time,high_school,no,travel,no
4,TR005,47,33.4,783,0.57,full_time,high_school,no,education,yes


In [18]:
CART=CARTTreeClassifier(max_depth=2,min_impurity_decrease=0.05)
CART.fit(df,y_col,x_cols,c_cols)
df['prediction']=CART.predict(df)
df_test['prediction']=CART.predict(df_test)
acc_train=np.mean(df['prediction']==df[y_col])
acc_test=np.mean(df_test['prediction']==df_test[y_col])
acc_train,acc_test

(0.9, 0.75)

'debt_ratio'